# OraGraphRAG Demo

Walks through ingesting a small folder, running three queries, and
visualizing the reweighted subgraph + amplitude heatmap.

Prerequisite: Oracle 23ai Free running (`docker compose up -d oracle-free`),
the schema initialized (`oragraphrag init-db --rebuild`), and a sample
folder ingested (`oragraphrag graphify tests/fixtures/sample_docs`).

In [ ]:
import asyncio
from pathlib import Path

import numpy as np

from oragraphrag.config import Config
from oragraphrag.embed import Embedder, build_axis_vectors
from oragraphrag.embed_backends import build_embed_backend
from oragraphrag.graph import GraphStore
from oragraphrag.llm import LLM
from oragraphrag.pipeline_query import QueryPipeline
from oragraphrag.viz import amplitude_heatmap_png, render_subgraph_html

cfg = Config.from_yaml("../config.yaml") if Path("../config.yaml").exists() else Config()
cfg

## Connect to Oracle 23ai and load ontology axis vectors

In [ ]:
store = GraphStore(cfg)
store.connect()

emb = Embedder(cfg, backend=build_embed_backend(cfg, store))
axes = await build_axis_vectors(emb)
print("axes loaded:", list(axes.keys()))

## Run three queries with different ontology-axis intents

Each query is designed to align with a different axis. The amplitude
heatmap below should show distinct activation patterns per question.

In [ ]:
questions = [
    "What causes ORA-21560 errors?",  # causal
    "What is HNSW?",  # definitional / taxonomic
    "When was the VECTOR datatype introduced?",  # temporal
]

async def run_all():
    async with LLM(cfg) as llm:
        pipeline = QueryPipeline(
            cfg=cfg, graph=store, embedder=emb, llm=llm, axis_vectors=axes
        )
        return [await pipeline.query(q) for q in questions]

results = await run_all()
for q, r in zip(questions, results):
    print(f"\n{q}")
    print("  amplitudes:", {k: round(v, 3) for k, v in r.amplitudes.items()})
    print("  answer:", r.answer.text[:120])

## Amplitude heatmap across questions

In [ ]:
amplitude_heatmap_png(
    [(q, r.amplitudes) for q, r in zip(questions, results)],
    out_path=Path("../paper/figures/edge_amplitude_heatmap.pdf"),
)
from IPython.display import Image

amplitude_heatmap_png(
    [(q, r.amplitudes) for q, r in zip(questions, results)],
    out_path=Path("./amplitude_heatmap.png"),
)
Image("./amplitude_heatmap.png")

## Per-query subgraph visualization

Edge thickness shows the reweighted edge weight (causal edges should
dominate the first query, definitional/taxonomic the second, temporal
the third).

In [ ]:
for q, r in zip(questions, results):
    out = Path(f"./subgraph_{q[:20].replace(' ', '_')}.html")
    activations = {}  # the orchestrator doesn't currently expose them; would need a small refactor
    render_subgraph_html(r.edges_used, activations=activations, out_path=out)
    print(f"wrote {out}")

In [ ]:
store.close()